# Lezione 5: La Maschera di Attività Regolatoria Contestuale
**File**: [`models/regulatory_mask.py`](file:///g:/My%20Drive/PhD%20&%20Research/Temporal%20GNN/cfm_grn/models/regulatory_mask.py)  
**Ruolo nel progetto**: Modella l'attivazione o repressione dinamica degli archi della rete di regolazione genica tramite la matrice di adiacenza efficace $A_{\text{eff}} = A_{\text{prior}} \odot M_\theta$.


---
## 1. Motivazione Biologica: Perché una Rete Statica Non Basta?

Nei database biologici (come ENCODE, Jaspar o ChIP-seq), un arco tra un Fattore di Trascrizione $j$ e un gene bersaglio $i$ indica solo che il legame fisico è **possibile**.
Tuttavia, in vivo:
1. **Specificità di Contesto**: Se il TF $j$ non è espresso o la cromatina è condensata, il legame non ha alcun effetto.
2. **Risposta alla Perturbazione**: Sotto stress farmacologico o knockout genetico, intere cascate di segnalazione vengono soppresse o attivate.

Definiamo quindi la **Matrice di Adiacenza Efficace**:
$$A_{\text{eff}}(x_t, t, c) = A_{\text{prior}} \odot M_\theta(x_t, t, c)$$
dove $M_\theta \in [0, 1]^E$ assegna un peso continuo di efficacia a ciascun arco noto, condizionato dallo stato attuale della cellula $x_t$, dal tempo $t$ e dalla perturbazione $c$.


---
## 2. Setup e Import


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from models.regulatory_mask import SparseAdjacencyPrior, RegulatoryActivityMask
from data.synthetic_grn import SyntheticGRNSimulator
print('Moduli regulatory_mask caricati!')


---
## 3. La Classe `SparseAdjacencyPrior`

Memorizza la topologia a priori in formato sparso `edge_index` di dimensione $[2, E]$, evitando l'allocazione quadratica di matrici dense $[G, G]$.


In [ ]:
sim = SyntheticGRNSimulator(num_genes=10, density=0.3, seed=42)
prior = SparseAdjacencyPrior(num_genes=sim.num_genes, edge_index=sim.edge_index)
print(f'Geni: {prior.num_genes}, Archi noti a priori: {prior.num_edges}')
print('Primi 5 archi (TF -> Bersaglio):\n', prior.edge_index[:, :5])


---
## 4. Architettura di `RegulatoryActivityMask`

La rete opera in due fasi:
1. **Cell State Encoder**: proietta $[x_t, t, c]$ in un vettore di contesto cellulare globale $h_{\text{cell}} \in \mathbb{R}^{d_{\text{ctx}}}$.
2. **Edge Scorer**: combina $h_{\text{cell}}$ con gli embedding statici del TF e del bersaglio per calcolare un logit sigmoideo per ciascun arco:
$$M_\theta(j \to i) = \sigma\Big(\text{MLP}\big([h_{\text{cell}}, e_j, e_i]\big)\Big) \in [0, 1]$$


In [ ]:
mask_net = RegulatoryActivityMask(
    num_genes=sim.num_genes,
    edge_index=sim.edge_index,
    perturbation_dim=sim.num_genes,
    hidden_dim=32,
    context_dim=32,
)
print(mask_net)


In [ ]:
# Test di forward pass
B = 4
x_dummy = torch.rand(B, sim.num_genes)
t_dummy = torch.full((B, 1), 0.5)
pert_dummy = torch.zeros(B, sim.num_genes)
pert_dummy[0, 0] = 1.0  # Cellula 0 ha knockout sul Gene 0

weights = mask_net(x_dummy, t_dummy, pert_dummy)
print('Shape dei pesi calcolati:', weights.shape)  # Deve essere [B, E]
print('Range valori (min, max):', weights.min().item(), weights.max().item())
assert (weights >= 0.0).all() and (weights <= 1.0).all(), 'I pesi devono essere in [0, 1]!'


---
## 5. Regolarizzazione L1 di Sparsità

Perché le reti biologiche sono naturalmente sparse, incentiviamo il modello a non attivare tutti gli archi simultaneamente aggiungendo una loss di sparsità L1:
$$\mathcal{L}_{L1} = \lambda \cdot \frac{1}{E} \sum_{e=1}^E M_{\theta, e}$$


In [ ]:
reg_loss = mask_net.compute_regularization_loss(weights, l1_weight=1e-3)
print(f'Perdita di regolarizzazione L1: {reg_loss.item():.6f}')
